# 01 · Başlangıç: veriyi indir, tanı, ilk model

**Proje kodu:** BVYZ-XX · **Öğrenci:** Ad Soyad

Bu defter projenin ilk teslimi için iskelettir. Üç şey yapar: veriyi indirir, tanır, sızıntısız bir temel model kurar.
`...` ile işaretli yerleri kendi veri setinize göre doldurun. Her hücrenin üstündeki not ne yapmanız gerektiğini söyler.

In [ ]:
SEED = 42
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
np.random.seed(SEED)

## 1 · Veriyi indir
Bu hücre, depoyu açan herkesin veriyi sıfırdan indirebilmesini sağlar. `data/` klasörüne yazar; depoya veri girmez (`.gitignore`).

In [ ]:
import os, urllib.request
os.makedirs("data", exist_ok=True)
URL = "..."                       # veri setinin doğrudan bağlantısı (ZIP ise açma adımını ekleyin)
DOSYA = "data/veri.csv"
if not os.path.exists(DOSYA):
    urllib.request.urlretrieve(URL, DOSYA)
df = pd.read_csv(DOSYA)
print(df.shape); df.head()

## 2 · Veriyi tanı (Hafta 1–3)
Kaç satır, kaç sütun, hedef ne, sınıf oranı kaç? Sayı mı kategori mi? Eksikler nerede ve rastgele mi?
**Bir hastanın kaç satırı var?** Bu sorunun cevabı bölme stratejinizi belirler.

In [ ]:
HEDEF = "..."
GRUP = None          # aynı hastanın birden çok satırı varsa hasta kimliği sütunu, yoksa None
y = df[HEDEF]; X = df.drop(columns=[HEDEF] + ([GRUP] if GRUP else []))
print("Hedef dağılımı:\n", y.value_counts(normalize=True).round(3))
print("\nEksik oranı (ilk 10):\n", X.isna().mean().sort_values(ascending=False).head(10).round(3))

## 3 · Sızıntısız temel model (Hafta 2 ve 4)
Her ön işleme adımı `Pipeline` içinde. Grup varsa `GroupKFold`, yoksa `StratifiedKFold`. Temel model lojistik regresyon; ileri modelleri sonraki defterde bununla karşılaştırın.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, GroupKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

sayisal = X.select_dtypes("number").columns.tolist()
kategorik = [c for c in X.columns if c not in sayisal]
on_isleme = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median", add_indicator=True)), ("sc", StandardScaler())]), sayisal),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("oh", OneHotEncoder(handle_unknown="ignore"))]), kategorik),
])
model = Pipeline([("on", on_isleme), ("lr", LogisticRegression(max_iter=3000))])
cv = GroupKFold(5) if GRUP else StratifiedKFold(5, shuffle=True, random_state=SEED)
p = cross_val_predict(model, X, y, cv=cv, groups=df[GRUP] if GRUP else None, method="predict_proba")[:, 1]
print(f"ROC-AUC {roc_auc_score(y, p):.3f} | PR-AUC {average_precision_score(y, p):.3f} | Brier {brier_score_loss(y, p):.3f}")

## 4 · Dört grafik (Hafta 4)
ROC, PR, kalibrasyon ve karar eğrisi. Bu dört grafik raporunuzun Sonuçlar bölümünün iskeleti; `rapor/sekiller/` altına kaydedin.

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve
from sklearn.calibration import calibration_curve
fpr, tpr, _ = roc_curve(y, p); pr, rc, _ = precision_recall_curve(y, p); g, t = calibration_curve(y, p, n_bins=8, strategy="quantile")
fig, ax = plt.subplots(1, 3, figsize=(13, 3.8))
ax[0].plot(fpr, tpr); ax[0].plot([0, 1], [0, 1], "--", c="gray"); ax[0].set_title("ROC")
ax[1].plot(rc, pr); ax[1].axhline(y.mean(), ls="--", c="gray"); ax[1].set_title("PR")
ax[2].plot(t, g, "o-"); ax[2].plot([0, 1], [0, 1], "--", c="gray"); ax[2].set_title("Kalibrasyon")
plt.tight_layout(); os.makedirs("rapor/sekiller", exist_ok=True); plt.savefig("rapor/sekiller/temel_model.png", dpi=150)

## 5 · Teslim 1 için kontrol listesi
- [ ] `data/README.md` indirme talimatını içeriyor, veri dosyası depoda değil
- [ ] README'de klinik soru ve veri tablosu dolu
- [ ] Bu defter sıfırdan (Çalışma zamanı → Tümünü çalıştır) hatasız çalışıyor
- [ ] Bölme stratejisi (grup var mı?) README'de yazılı